In [2]:
import pandas as pd
import os 

from sqlalchemy import create_engine

In [3]:
engine = create_engine(
    "mssql+pyodbc://localhost/premier_league_db?driver=ODBC+Driver+17+for+SQL+Server"
)

print("Connected successfully!")

Connected successfully!


In [4]:
folder_path = "../data/raw"

csv_files = [
    file for file in os.listdir(folder_path)
    if file.endswith(".csv")
]

print(csv_files)

['21-22.csv', '22-23.csv', '23-24.csv', '24-25.csv', '25-26.csv']


In [5]:
all_dataframes = []

for file in csv_files:

    file_path = os.path.join(folder_path, file)

    df = pd.read_csv(file_path)

    season = file.replace(".csv", "")

    df["season"] = season

    all_dataframes.append(df)

print("All files loaded!")

All files loaded!


In [6]:
combined_df = pd.concat(
    all_dataframes,
    ignore_index=True
)

combined_df.head()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
0,E0,13/08/2021,20:00,Brentford,Arsenal,2,0,H,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,E0,14/08/2021,12:30,Man United,Leeds,5,1,H,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,E0,14/08/2021,15:00,Burnley,Brighton,1,2,A,1,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,E0,14/08/2021,15:00,Chelsea,Crystal Palace,3,0,H,2,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,E0,14/08/2021,15:00,Everton,Southampton,3,1,H,0,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [7]:
combined_df.tail()

,Div,Date,Time,HomeTeam,AwayTeam,FTHG,FTAG,FTR,HTHG,HTAG,...,BMGMCA,BVCH,BVCD,BVCA,CLCH,CLCD,CLCA,LBCH,LBCD,LBCA
1874,E0,10/05/2026,14:00,Burnley,Aston Villa,2,2,D,1,1,...,1.45,7.00,4.50,1.45,NaN,NaN,NaN,6.00,4.33,1.50
1875,E0,10/05/2026,14:00,Crystal Palace,Everton,2,2,D,1,1,...,2.55,2.88,3.13,2.60,NaN,NaN,NaN,2.70,3.30,2.60
1876,E0,10/05/2026,14:00,Nott'm Forest,Newcastle,1,1,D,0,0,...,2.20,3.40,3.50,2.10,NaN,NaN,NaN,3.25,3.40,2.20
1877,E0,10/05/2026,16:30,West Ham,Arsenal,0,1,A,0,0,...,1.50,5.50,4.00,1.60,NaN,NaN,NaN,5.50,4.00,1.57
1878,E0,11/05/2026,20:00,Tottenham,Leeds,1,1,D,0,0,...,4.35,1.83,3.75,4.10,1.83,3.7,4.2,1.83,3.70,4.20


#### Next step after creating tables in SQL SEREVER 

In [8]:
home_teams = combined_df["HomeTeam"]

away_teams = combined_df["AwayTeam"]

all_teams = pd.concat([home_teams, away_teams])

unique_teams = pd.DataFrame(
    all_teams.unique(),
    columns=["team_name"]
)

unique_teams.head()

,team_name
0,Brentford
1,Man United
2,Burnley
3,Chelsea
4,Everton


In [9]:
unique_teams.tail()

,team_name
22,Nott'm Forest
23,Sheffield United
24,Luton
25,Ipswich
26,Sunderland


In [10]:
unique_teams.count

<bound method DataFrame.count of            team_name
0          Brentford
1         Man United
2            Burnley
3            Chelsea
4            Everton
5          Leicester
6            Watford
7            Norwich
8          Newcastle
9          Tottenham
10         Liverpool
11       Aston Villa
12    Crystal Palace
13             Leeds
14          Man City
15          Brighton
16       Southampton
17            Wolves
18           Arsenal
19          West Ham
20            Fulham
21       Bournemouth
22     Nott'm Forest
23  Sheffield United
24             Luton
25           Ipswich
26        Sunderland>

In [ ]:
unique_teams.to_sql(
    "teams",
    engine,
    if_exists="append",
    index=False
)

print("Teams table loaded successfully!")

In [13]:
teams_df = pd.read_sql(
    "SELECT * FROM teams",
    engine
)

teams_df.head()

,team_id,team_name
0,19,Arsenal
1,12,Aston Villa
2,22,Bournemouth
3,1,Brentford
4,16,Brighton


In [14]:
team_lookup = dict(
    zip(
        teams_df["team_name"],
        teams_df["team_id"]
    )
)

team_lookup

{'Arsenal': 19,
 'Aston Villa': 12,
 'Bournemouth': 22,
 'Brentford': 1,
 'Brighton': 16,
 'Burnley': 3,
 'Chelsea': 4,
 'Crystal Palace': 13,
 'Everton': 5,
 'Fulham': 21,
 'Ipswich': 26,
 'Leeds': 14,
 'Leicester': 6,
 'Liverpool': 11,
 'Luton': 25,
 'Man City': 15,
 'Man United': 2,
 'Newcastle': 9,
 'Norwich': 8,
 "Nott'm Forest": 23,
 'Sheffield United': 24,
 'Southampton': 17,
 'Sunderland': 27,
 'Tottenham': 10,
 'Watford': 7,
 'West Ham': 20,
 'Wolves': 18}

In [15]:
matches_df = pd.DataFrame()

matches_df["match_date"] = pd.to_datetime(
    combined_df["Date"],
    dayfirst=True
)

matches_df["season"] = combined_df["season"]

matches_df["home_team_id"] = combined_df["HomeTeam"].map(team_lookup)

matches_df["away_team_id"] = combined_df["AwayTeam"].map(team_lookup)

matches_df["home_goals"] = combined_df["FTHG"]

matches_df["away_goals"] = combined_df["FTAG"]

matches_df["result"] = combined_df["FTR"]

matches_df.head()

,match_date,season,home_team_id,away_team_id,home_goals,away_goals,result
0,2021-08-13,21-22,1,19,2,0,H
1,2021-08-14,21-22,2,14,5,1,H
2,2021-08-14,21-22,3,16,1,2,A
3,2021-08-14,21-22,4,13,3,0,H
4,2021-08-14,21-22,5,17,3,1,H


In [19]:
matches_df.tail()

,match_date,season,home_team_id,away_team_id,home_goals,away_goals,result
1874,2026-05-10,25-26,3,12,2,2,D
1875,2026-05-10,25-26,13,5,2,2,D
1876,2026-05-10,25-26,23,9,1,1,D
1877,2026-05-10,25-26,20,19,0,1,A
1878,2026-05-11,25-26,10,14,1,1,D


In [20]:
matches_df.isnull().sum()

match_date      0
season          0
home_team_id    0
away_team_id    0
home_goals      0
away_goals      0
result          0
dtype: int64

In [21]:
matches_df.dtypes

match_date      datetime64[ns]
season                  object
home_team_id             int64
away_team_id             int64
home_goals               int64
away_goals               int64
result                  object
dtype: object

In [24]:
print(matches_df.columns.tolist())

['match_date', 'season', 'home_team_id', 'away_team_id', 'home_goals', 'away_goals', 'result']


In [25]:
matches_df.to_sql(
    "matches",
    engine,
    if_exists="append",
    index=False
)
print("Matches table loaded successfully!")

Matches table loaded successfully!
